# Week 4 — Baseline Model (revised)

This notebook trains a plain `LinearRegression` on `outputs/02_preprocessed_v2.csv`, using the shared train-only preprocessing in `model_pipeline.py`.
Decisions are made on validation (training months ending 2026-03, scored on 2026-04). The May 2026 test month is scored once, for the final model only.

## Why the first baseline scored R² = 0.23

1. Three training sales of $645M–$796M (likely data-entry errors) dominated the squared-error fit.
2. Implausible feature values (LivingArea of 0, LotSize up to about 2.1B sqft, 175 bathrooms) were not capped.
3. There were no location features, even though ZIP code drives price and is almost never null.
4. The raw-dollar target is heavily right-skewed, which a linear fit on size cannot capture well.
5. The training window was chosen with a median-price baseline rather than the model itself.
6. The two one-hot columns were constant, so they added nothing.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import model_pipeline as mp

INPUT_FILE = Path('outputs/02_preprocessed_v2.csv')
ORIGINAL_TEST_R2 = 0.2295   # v1 baseline, from the earlier run of this notebook
MIN_FREQUENCY = 5
FEATURES = mp.NUMERIC_FEATURES + mp.LOCATION_CATEGORICAL
BASE4 = ['LivingArea', 'Bedrooms', 'Bathrooms', 'LotSize']

if not INPUT_FILE.exists():
    raise FileNotFoundError(f'{INPUT_FILE} was not found. Run 02_preprocessing.ipynb first.')
data = pd.read_csv(INPUT_FILE, dtype={'PostalCode': 'object'})
val_train_all = data.loc[data['val_split'].eq('train')]
val_rows = data.loc[data['val_split'].eq('val')]
train_all = data.loc[data['split'].eq('train')]
test_rows = data.loc[data['split'].eq('test')]
val_train, val_bounds = mp.filter_train_prices(val_train_all)
train, train_bounds = mp.filter_train_prices(train_all)
print(f'Validation phase: train {len(val_train):,} rows ({val_train["Month"].min()}..{val_train["Month"].max()}), val {len(val_rows):,} rows')
print(f'Final: train {len(train):,} rows ({train["Month"].min()}..{train["Month"].max()}), test {len(test_rows):,} rows')
print(f'Train price bounds kept: ${train_bounds[0]:,.0f} - ${train_bounds[1]:,.0f} ({len(train_all) - len(train):,} train rows dropped)')

Validation phase: train 127,734 rows (2025-04..2026-03), val 12,021 rows
Final: train 127,939 rows (2025-05..2026-04), test 12,015 rows
Train price bounds kept: $235,000 - $8,900,000 (1,894 train rows dropped)


In [2]:
def original_pipeline():
    numeric = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler())])
    return Pipeline([('prep', ColumnTransformer([('num', numeric, BASE4)])), ('model', LinearRegression())])

last_month = val_train_all['Month'].max()
steps = [
    ('v1 original (4 features, 1 month, no filters, $ target)', original_pipeline(),
     val_train_all.loc[val_train_all['Month'].eq(last_month)], BASE4),
    ('a) + train price filter + feature clipping', mp.make_linear_model(False, numeric=BASE4, log_features=[], categorical=[]),
     val_train, BASE4),
    ('b) + structural fields + ZIP one-hot', mp.make_linear_model(False, log_features=[], min_frequency=MIN_FREQUENCY),
     val_train, FEATURES),
    ('c+d) + log target, selected window (final)', mp.make_linear_model(True, log_features=[], min_frequency=MIN_FREQUENCY),
     val_train, FEATURES),
]
val_table = []
for name, model, fit_rows, cols in steps:
    model.fit(fit_rows[cols], fit_rows[mp.TARGET])
    val_table.append({'step': name, **mp.regression_metrics(val_rows[mp.TARGET], model.predict(val_rows[cols]))})
val_table = pd.DataFrame(val_table).set_index('step')
val_table.style.format({'R2': '{:.4f}', 'MAE': '${:,.0f}', 'MdAPE': '{:.1%}'})

,R2,MAE,MdAPE
step,,,
"v1 original (4 features, 1 month, no filters, $ target)",0.0056,"$807,415",41.6%
a) + train price filter + feature clipping,0.0037,"$770,166",37.1%
b) + structural fields + ZIP one-hot,0.0076,"$489,017",16.6%
"c+d) + log target, selected window (final)",0.0095,"$422,895",9.8%


In [3]:
final_model = mp.make_linear_model(True, log_features=[], min_frequency=MIN_FREQUENCY)
final_model.fit(train[FEATURES], train[mp.TARGET])
test_metrics = mp.regression_metrics(test_rows[mp.TARGET], final_model.predict(test_rows[FEATURES]))
print(f'Test rows scored: {len(test_rows):,} (all valid May 2026 rows, none dropped by price)')
print(f"Test R²: {test_metrics['R2']:.4f} | MAE: ${test_metrics['MAE']:,.0f} | MdAPE: {test_metrics['MdAPE']:.1%}")

Test rows scored: 12,015 (all valid May 2026 rows, none dropped by price)
Test R²: 0.4860 | MAE: $235,941 | MdAPE: 9.8%


In [4]:
summary = pd.DataFrame({
    'validation R2': [val_table['R2'].iloc[0], val_table['R2'].iloc[-1]],
    'validation MAE': [val_table['MAE'].iloc[0], val_table['MAE'].iloc[-1]],
    'test R2': [ORIGINAL_TEST_R2, test_metrics['R2']],
    'test MAE': [float('nan'), test_metrics['MAE']],
    'test MdAPE': [float('nan'), test_metrics['MdAPE']],
}, index=['before (v1 baseline)', 'after (revised baseline)'])
summary.style.format({'validation R2': '{:.4f}', 'validation MAE': '${:,.0f}', 'test R2': '{:.4f}',
                      'test MAE': '${:,.0f}', 'test MdAPE': '{:.1%}'}, na_rep='n/a')

,validation R2,validation MAE,test R2,test MAE,test MdAPE
before (v1 baseline),0.0056,"$807,415",0.2295,n/a,n/a
after (revised baseline),0.0095,"$422,895",0.4860,"$235,941",9.8%


## Baseline result

| | Validation R² | Validation MAE | Test R² | Test MAE | Test MdAPE |
|---|---|---|---|---|---|
| Before (v1 baseline) | 0.0056 | $807,415 | **0.2295** | n/a | n/a |
| After (revised baseline) | 0.0095 | $422,895 | **0.4860** | $235,941 | 9.8% |

- **Final model:** plain `LinearRegression` on `log(ClosePrice)`, trained on May 2025–April 2026 (12-month window chosen by validation MAE, 127,939 rows after the train-only price filter of \$235k–\$8.9M).
- **Test set:** all 12,015 valid May 2026 sales. 9 rows were excluded only because `LivingArea` was missing or under 200 sqft; no test row was dropped because of price.
- **Validation R² is uninformative:** three April 2026 sales of \$645M–\$796M account for about 98% of April's total variance. Validation MAE and MdAPE were used for decisions instead; MAE was nearly halved and MdAPE dropped from 41.6% to 9.8%.
- **Biggest gains:** ZIP-code location and the log target. Price filtering on its own barely moves the metrics, but it keeps the fit from being driven by erroneous records.
- **Remaining gap:** test R² is still pulled down by a handful of \$20M–\$98M luxury sales. Week 5 will compare tree models on this exact data and split.